# Executed Pipeline Evidence

This notebook reads tracked artifacts produced by the Lab 04 pipeline. Its code is read-only, so it can be executed safely without live infrastructure.

In [1]:
from pathlib import Path
import json

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(path for path in candidates if (path / 'data/processed/discovered_files.json').exists())
print(f'Artifact root: {PROJECT_ROOT.name}')

Artifact root: Streaming


## Repository discovery

In [2]:
manifest = json.loads((PROJECT_ROOT / 'data/processed/discovered_files.json').read_text())
print('repo_path:', manifest['repo_path'])
print('total_files:', manifest['total_files'])
print('first_5_files:')
for file_path in manifest['files'][:5]:
    print('-', file_path)

repo_path: data/raw/accelerate
total_files: 99
first_5_files:
- benchmarks/big_model_inference/big_model_inference.py
- benchmarks/big_model_inference/measures_util.py
- benchmarks/fp8/ms_amp/ddp.py
- benchmarks/fp8/ms_amp/distrib_deepspeed.py
- benchmarks/fp8/ms_amp/fp8_utils.py


## Kafka topic/event samples

In [3]:
sample_dir = PROJECT_ROOT / 'evidence/kafka'
for name in ['nodes_sample.json', 'edges_sample.json', 'metadata_sample.json', 'errors_sample.json']:
    captured = (sample_dir / name).read_text().strip()
    # Capture files may contain either raw JSON or Kafka key\tvalue text.
    payload = captured.partition('\\t')[2] or captured.partition('\t')[2] or captured
    event = json.loads(payload)
    identity = next((event[key] for key in ('node_id', 'edge_id', 'metadata_id') if key in event), '<error-event>')
    print(f"{name}: schema={event.get('schema_version')} time={event.get('event_time')} id={identity}")

nodes_sample.json: schema=1.0 time=2026-07-05T05:33:57.750659Z id=38e41e11cb41c8fe7ec4ceb45a3aaa640e5639df205125aea4282b6d9e0e4a10
edges_sample.json: schema=1.0 time=2026-07-05T05:33:57.750659Z id=e5077591dc0c165d2a86d3476884c1127a3aed08154d380e8d0de8342ce4f376
metadata_sample.json: schema=1.0 time=2026-07-05T05:33:57.750659Z id=193be55d48d094dd2dabd8f933ffc82c76bef578804d6aa63503a24c22bb11d5
errors_sample.json: schema=1.0 time=2026-07-05T05:34:43.384711Z id=<error-event>


## MongoDB metadata verification

In [4]:
def matching_lines(path, terms, limit=14):
    lines = path.read_text(errors='replace').splitlines()
    selected = [line.strip() for line in lines if any(term.lower() in line.lower() for term in terms)]
    return selected[-limit:]

pipeline_log = PROJECT_ROOT / 'evidence/logs/terminal_2_pipeline_latest.log'
for line in matching_lines(pipeline_log, ['MongoDB', 'metadata', 'duplicate', 'Count delta']):
    print(line)

metadata_documents=+0
INFO     Duplicate count: metadata_id=0
INFO     Duplicate graph IDs: node_id=0 edge_id=0
INFO     MongoDB metadata reflects the replayed
[07/05/26 12:34:40] INFO     MongoDB metadata documents: 99
INFO     Duplicate metadata_id groups: 0
INFO     Duplicate repo/file groups: 0
INFO     Target metadata: {'event_time':
'metadata_id':
INFO     Duplicate node IDs: 0
INFO     Duplicate edge IDs: 0
│ ❱  53 │   │   nodes, edges, metadata │
│ ❱ 35 │   nodes, ast_edges, metadata  │
metadata_id=59f203db77b71174cb16b9dba979


## Neo4j graph verification

In [5]:
for line in matching_lines(pipeline_log, ['Neo4j totals', 'Duplicate node', 'Duplicate edge', 'placeholder', 'Target file']):
    print(line)

[07/05/26 12:34:26] INFO     Neo4j totals: nodes=263154 edges=626918
INFO     Duplicate node IDs: 0
INFO     Duplicate edge IDs: 0
INFO     Unresolved placeholder nodes: 0
[07/05/26 12:34:27] INFO     Target file
[07/05/26 12:34:42] INFO     Neo4j totals: nodes=263154 edges=626917
INFO     Duplicate node IDs: 0
INFO     Duplicate edge IDs: 0
INFO     Unresolved placeholder nodes: 0
[07/05/26 12:34:43] INFO     Target file


## Replay/checkpoint evidence

In [6]:
checkpoint_candidates = [
    PROJECT_ROOT / 'evidence/logs/checkpoint_resume.log',
    PROJECT_ROOT / 'book/logs/checkpoint_resume.log',
]
checkpoint_log = next((path for path in checkpoint_candidates if path.exists()), None)
if checkpoint_log:
    print(checkpoint_log.read_text().strip())
else:
    print('Checkpoint evidence not found in evidence/logs or book/logs.')

print('--- structured replay verification ---')
replay_log = PROJECT_ROOT / 'evidence/logs/identity_replay_verification.log'
replay_terms = [
    'target_file=', 'file_hash_changed=',
    'neo4j_target_nodes_before=', 'neo4j_target_nodes_after=',
    'neo4j_target_edges_before=', 'neo4j_target_edges_after=',
    'duplicate_node_id_groups=', 'duplicate_edge_id_groups=',
]
for line in replay_log.read_text().splitlines():
    if any(line.startswith(term) for term in replay_terms):
        print(line)

Checkpoint evidence not found in evidence/logs or book/logs.
--- structured replay verification ---
target_file=src/accelerate/_lab_replay_probe.py
file_hash_changed=True
neo4j_target_nodes_before=14
neo4j_target_nodes_after=14
neo4j_target_edges_before=27
neo4j_target_edges_after=26
duplicate_node_id_groups=0
duplicate_edge_id_groups=0


## Reflection

**What worked:** Stable event identities, connector-backed metadata upsert, checkpoint recovery, and duplicate verification are represented by reproducible artifacts.

**What failed:** Documentation previously drifted from the checked-in discovery manifest, and the book had no executed notebook evidence.

**How it was fixed:** The manifest description was synchronized and this read-only notebook was executed against tracked evidence.

**Remaining limitations:** CFG, DFG, and CALL construction remains lightweight. Modified-file graph replay uses a controlled direct Neo4j cleanup before updated events are republished; metadata replay remains event-driven through Kafka, checkpointed Spark Structured Streaming, and MongoDB.